In [9]:
import time
import math
from datetime import datetime, timedelta
from typing import Optional, Iterable, Union, List

import requests
import pandas as pd


def fetch_system_prices(
    start_date: Union[str, datetime],
    end_date: Union[str, datetime],
    pause_seconds: float = 0.35,
    max_retries: int = 4,
    save_csv_path: Optional[str] = None,
) -> pd.DataFrame:
    """
    Fetch BMRS System Prices from the new Elexon API for a date range, inclusive.
    Loops all 48 settlement periods per settlement date.

    Args:
        start_date: 'YYYY-MM-DD' or datetime for first settlement date (inclusive).
        end_date:   'YYYY-MM-DD' or datetime for last settlement date (inclusive).
        pause_seconds: polite delay between requests to avoid 429s.
        max_retries:   per-request retries with exponential backoff on 429/5xx.
        save_csv_path: if provided, writes the final DataFrame to this CSV path.

    Returns:
        pd.DataFrame with one row per (date, SP) record. Columns are whatever the
        endpoint returns (flattened). Numeric-looking fields are converted to float
        where possible.
    """
    def _to_date(d):
        if isinstance(d, str):
            return datetime.strptime(d, "%Y-%m-%d").date()
        if isinstance(d, datetime):
            return d.date()
        return d  # assume date

    start = _to_date(start_date)
    end = _to_date(end_date)
    if end < start:
        raise ValueError("end_date must be on/after start_date")

    base = "https://data.elexon.co.uk/bmrs/api/v1/balancing/settlement/system-prices/{date}/{sp}"

    session = requests.Session()
    headers = {"accept": "application/json"}

    def _get_with_retries(url: str) -> requests.Response:
        for attempt in range(max_retries):
            resp = session.get(url, headers=headers, timeout=30)
            if resp.status_code < 400:
                return resp

            # Retry only on 429 or 5xx
            if resp.status_code in (429, 500, 502, 503, 504):
                # exponential backoff with jitter
                sleep_for = (2 ** attempt) * 0.5 + (0.1 * math.sin(attempt))
                time.sleep(max(sleep_for, pause_seconds))
                continue

            # Hard error (4xx other than 429)
            resp.raise_for_status()

        # If we’re here, all retries failed
        resp.raise_for_status()
        return resp  # unreachable, but for type checkers

    records: List[dict] = []

    cur = start
    while cur <= end:
        date_str = cur.isoformat()
        for sp in range(1, 49):
            url = base.format(date=date_str, sp=sp)
            resp = _get_with_retries(url)

            payload = resp.json()

            # The API returns {"data":[{...}],"metadata":{...}} for success.
            data_block = payload.get("data", payload)

            # Some SPs may have empty data; skip gracefully.
            if not data_block:
                time.sleep(pause_seconds)
                continue

            # Ensure iterable
            if isinstance(data_block, dict):
                data_iter: Iterable[dict] = [data_block]
            else:
                data_iter = data_block

            for row in data_iter:
                # Make sure SettlementDate/SP present even if omitted
                row.setdefault("SettlementDate", date_str)
                row.setdefault("SettlementPeriod", sp)
                records.append(row)

            time.sleep(pause_seconds)

        cur += timedelta(days=1)

    if not records:
        return pd.DataFrame()

    df = pd.json_normalize(records)

    # Best-effort numeric conversion for numeric-looking columns
    for col in df.columns:
        # Skip obviously non-numeric
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors="ignore")

    # Optional: sort for sanity
    if "SettlementDate" in df.columns and "SettlementPeriod" in df.columns:
        df = df.sort_values(["SettlementDate", "SettlementPeriod"]).reset_index(drop=True)

    if save_csv_path:
        df.to_csv(save_csv_path, index=False)

    return df





In [10]:
import pandas as pd
import os

MASTER_FILE = "../data/elexon_full_prices.csv"

# Load existing if it exists, otherwise create new empty df
if os.path.exists(MASTER_FILE):
    df_old = pd.read_csv(MASTER_FILE)
else:
    df_old = pd.DataFrame()


In [12]:
df_new = fetch_system_prices(
    start_date="2025-02-01",
    end_date="2025-04-30",
    pause_seconds=0.4,
    max_retries=5
)

df_new.head(), df_new.shape

C:\Users\Iznaur\AppData\Local\Temp\ipykernel_60908\3128924444.py:114: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


(  settlementDate  settlementPeriod             startTime  \
 0     2025-02-01                 1  2025-02-01T00:00:00Z   
 1     2025-02-01                 2  2025-02-01T00:30:00Z   
 2     2025-02-01                 3  2025-02-01T01:00:00Z   
 3     2025-02-01                 4  2025-02-01T01:30:00Z   
 4     2025-02-01                 5  2025-02-01T02:00:00Z   
 
         createdDateTime  systemSellPrice  systemBuyPrice  bsadDefaulted  \
 0  2025-02-02T00:44:32Z            94.86           94.86          False   
 1  2025-02-02T01:14:45Z            93.10           93.10          False   
 2  2025-02-02T01:44:37Z            93.10           93.10          False   
 3  2025-02-02T02:14:49Z            93.10           93.10          False   
 4  2025-02-02T02:44:30Z            93.15           93.15          False   
 
   priceDerivationCode  reserveScarcityPrice  netImbalanceVolume  ...  \
 0                   N                   0.0         -146.910305  ...   
 1                   N      

In [13]:
# Standardise types
df_new["settlementDate"] = pd.to_datetime(df_new["settlementDate"])
df_old["settlementDate"] = pd.to_datetime(df_old["settlementDate"])

# Combine and remove duplicates
df_combined = pd.concat([df_old, df_new], ignore_index=True)

df_combined = df_combined.drop_duplicates(
    subset=["settlementDate", "settlementPeriod"],
    keep="last"
).sort_values(["settlementDate", "settlementPeriod"])


In [14]:
df_combined.to_csv(MASTER_FILE, index=False)
print("✅ Master dataset updated:", df_combined.shape)

✅ Master dataset updated: (11566, 24)
